[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_04_pytree_ops_solution.ipynb)

# 🟡 Solution: Stack a List of Pytrees

*JAX Fundamentals · Medium*

Reference implementation. Try it yourself in `b_04_pytree_ops.ipynb` first.

---
Given a **list of pytrees** that all share the same structure, produce a single
pytree of the same structure where each leaf is the `jnp.stack` of the
corresponding leaves.

```
[{"w": (3, 2), "b": (2,)},        ->   {"w": (4, 3, 2), "b": (4, 2)}
 {"w": (3, 2), "b": (2,)},
 {"w": (3, 2), "b": (2,)},
 {"w": (3, 2), "b": (2,)}]
```

### Rules
- Handle arbitrary nesting: dicts, lists, tuples, and mixtures of them
- Raise `ValueError` if the input list is empty
- Raise `ValueError` if the trees do not all share the same structure
- No hand-written recursion over dicts — use the `jax.tree` utilities

### Why it matters
This is how you build an **ensemble**: stack N independently-initialised
parameter sets into one pytree, then `vmap` your model over the leading axis to
run all N models in a single batched call. The same trick collects per-step
metrics from a training loop into arrays, and assembles the `xs` argument for
`lax.scan`.

Interviewers like it because the naive answer is a pile of nested loops, and the
JAX answer is one line of `tree.map` with a variadic lambda.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def tree_stack(trees):
    trees = list(trees)
    if not trees:
        raise ValueError("tree_stack requires at least one tree")

    ref = jax.tree.structure(trees[0])
    for i, t in enumerate(trees[1:], start=1):
        s = jax.tree.structure(t)
        if s != ref:
            raise ValueError(
                f"tree {i} has structure {s}, expected {ref}"
            )

    # tree.map walks all trees in lockstep; *leaves collects one leaf per tree.
    return jax.tree.map(lambda *leaves: jnp.stack(leaves), *trees)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

# Three "models", each a small parameter pytree.
models = [
    {"w": jnp.full((3, 2), float(i)), "b": jnp.full((2,), float(i))}
    for i in range(3)
]

stacked = tree_stack(models)
print("leaf shapes:", jax.tree.map(lambda a: a.shape, stacked))
print("stacked['b']:\n", stacked["b"])

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("pytree_ops")